# EduPredict-XAI
## Step 14 — Train/Test Split and Model-Specific Preprocessing

**Research Topic:** Student Performance Prediction Using Explainable AI  
**Student:** Payal Pramod Pawar  
**Program:** MCA Sem 3  
**Guide:** Mrs. Shaesta Mujawar  

### Objective

The objective of this notebook is to prepare the feature-engineered student performance dataset for machine learning.

The process includes:

- Loading the feature-engineered dataset
- Separating features and target
- Creating training and testing datasets
- Identifying numerical and categorical features
- Preparing model-specific preprocessing
- Preventing data leakage during preprocessing
- Preparing datasets for CatBoost, XGBoost, LightGBM and TabPFN

In [1]:
# Import libraries
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Set project paths
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

print("Project root:")
print(PROJECT_ROOT)

print("\nProcessed data folder:")
print(PROCESSED_DATA_PATH)

Project root:
d:\MCA Sem 3\RP\EduPredict-XAI

Processed data folder:
d:\MCA Sem 3\RP\EduPredict-XAI\data\processed


In [3]:
# Load the feature-engineered dataset
feature_engineered_path = (
    PROCESSED_DATA_PATH /
    "student_performance_feature_engineered.csv"
)

df = pd.read_csv(feature_engineered_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Shape: (1000, 10)


,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score
0,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0
1,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0
2,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3
3,Male,2.6,77.5,8.0,High School,Yes,Yes,No,85.1,83.8
4,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3


In [4]:
# Check columns
print("Dataset columns:")

for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")

Dataset columns:
1. gender
2. study_time_hours
3. attendance_percent
4. sleep_hours
5. parental_education
6. internet_access
7. extracurricular_activities
8. part_time_job
9. previous_grade
10. final_exam_score


In [5]:
# Define the target
TARGET = "final_exam_score"

print("Target variable:", TARGET)
print("Target exists:", TARGET in df.columns)

Target variable: final_exam_score
Target exists: True


In [6]:
# Separate X and y
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1000, 9)
Target shape: (1000,)


In [7]:
# Verify the features
print("Features being used by the models:\n")

for i, feature in enumerate(X.columns, start=1):
    print(f"{i}. {feature}")

Features being used by the models:

1. gender
2. study_time_hours
3. attendance_percent
4. sleep_hours
5. parental_education
6. internet_access
7. extracurricular_activities
8. part_time_job
9. previous_grade


In [8]:
# Identify numerical and categorical features
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical features:")
for feature in numeric_features:
    print("→", feature)

print("\nCategorical features:")
for feature in categorical_features:
    print("→", feature)

Numerical features:
→ study_time_hours
→ attendance_percent
→ sleep_hours
→ previous_grade

Categorical features:
→ gender
→ parental_education
→ internet_access
→ extracurricular_activities
→ part_time_job


In [9]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Training data:
X_train: (800, 9)
y_train: (800,)

Testing data:
X_test: (200, 9)
y_test: (200,)


In [11]:
# Model 1 — CatBoost preprocessing
X_train_catboost = X_train.copy()
X_test_catboost = X_test.copy()

print("CatBoost training shape:", X_train_catboost.shape)
print("CatBoost testing shape:", X_test_catboost.shape)

for col in categorical_features:
    X_train_catboost[col] = X_train_catboost[col].astype(str)
    X_test_catboost[col] = X_test_catboost[col].astype(str)

print("Categorical columns prepared for CatBoost.")

CatBoost training shape: (800, 9)
CatBoost testing shape: (200, 9)
Categorical columns prepared for CatBoost.


In [12]:
# Save CatBoost categorical feature positions
catboost_cat_features = [
    X_train_catboost.columns.get_loc(col)
    for col in categorical_features
]

print("Categorical column indices for CatBoost:")
print(catboost_cat_features)

Categorical column indices for CatBoost:
[0, 4, 5, 6, 7]


In [13]:
# Model 2 — XGBoost / LightGBM preprocessing
# Create encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            encoder,
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [14]:
# Fit ONLY on training data
X_train_encoded = preprocessor.fit_transform(X_train)

X_test_encoded = preprocessor.transform(X_test)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)

Encoded training shape: (800, 16)
Encoded testing shape: (200, 16)


In [15]:
# Get encoded feature names
encoded_feature_names = (
    preprocessor
    .get_feature_names_out()
)

print("Encoded feature names:")

for feature in encoded_feature_names:
    print("→", feature)

Encoded feature names:
→ categorical__gender_Female
→ categorical__gender_Male
→ categorical__parental_education_Bachelors
→ categorical__parental_education_High School
→ categorical__parental_education_Masters
→ categorical__parental_education_PhD
→ categorical__internet_access_No
→ categorical__internet_access_Yes
→ categorical__extracurricular_activities_No
→ categorical__extracurricular_activities_Yes
→ categorical__part_time_job_No
→ categorical__part_time_job_Yes
→ remainder__study_time_hours
→ remainder__attendance_percent
→ remainder__sleep_hours
→ remainder__previous_grade


In [16]:
# Convert encoded data into DataFrames
X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

print("Training encoded DataFrame:")
display(X_train_encoded_df.head())

Training encoded DataFrame:


,categorical__gender_Female,categorical__gender_Male,categorical__parental_education_Bachelors,categorical__parental_education_High School,categorical__parental_education_Masters,categorical__parental_education_PhD,categorical__internet_access_No,categorical__internet_access_Yes,categorical__extracurricular_activities_No,categorical__extracurricular_activities_Yes,categorical__part_time_job_No,categorical__part_time_job_Yes,remainder__study_time_hours,remainder__attendance_percent,remainder__sleep_hours,remainder__previous_grade
29,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,2.4,94.3,7.1,86.8
535,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,2.0,92.5,7.7,58.7
695,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,3.0,91.0,7.0,56.6
557,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,3.4,81.6,6.6,47.4
836,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,4.5,92.4,8.4,90.7


In [17]:
# Check for missing values
print(
    "Missing values in encoded training data:",
    X_train_encoded_df.isnull().sum().sum()
)

print(
    "Missing values in encoded testing data:",
    X_test_encoded_df.isnull().sum().sum()
)

Missing values in encoded training data: 0
Missing values in encoded testing data: 0


In [18]:
# Prepare XGBoost data
X_train_xgb = X_train_encoded_df.copy()
X_test_xgb = X_test_encoded_df.copy()

print("XGBoost training shape:", X_train_xgb.shape)
print("XGBoost testing shape:", X_test_xgb.shape)

XGBoost training shape: (800, 16)
XGBoost testing shape: (200, 16)


In [22]:
# Prepare LightGBM data
X_train_lgbm = X_train_encoded_df.copy()
X_test_lgbm = X_test_encoded_df.copy()

print("LightGBM training shape:", X_train_lgbm.shape)
print("LightGBM testing shape:", X_test_lgbm.shape)

LightGBM training shape: (800, 16)
LightGBM testing shape: (200, 16)


In [23]:
# Now prepare TabPFN
X_train_tabpfn = X_train.copy()
X_test_tabpfn = X_test.copy()

y_train_tabpfn = y_train.copy()
y_test_tabpfn = y_test.copy()

print("TabPFN training shape:", X_train_tabpfn.shape)
print("TabPFN testing shape:", X_test_tabpfn.shape)

TabPFN training shape: (800, 9)
TabPFN testing shape: (200, 9)


In [24]:
# Final verification cell
print("=" * 60)
print("STEP 14 — FINAL DATA PREPARATION SUMMARY")
print("=" * 60)

print("\nOriginal dataset:")
print("X:", X.shape)
print("y:", y.shape)

print("\nTrain/Test split:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nFeature types:")
print("Numerical:", len(numeric_features))
print("Categorical:", len(categorical_features))

print("\nModel-specific data:")
print("CatBoost:", X_train_catboost.shape)
print("XGBoost:", X_train_xgb.shape)
print("LightGBM:", X_train_lgbm.shape)
print("TabPFN:", X_train_tabpfn.shape)

STEP 14 — FINAL DATA PREPARATION SUMMARY

Original dataset:
X: (1000, 9)
y: (1000,)

Train/Test split:
X_train: (800, 9)
X_test: (200, 9)
y_train: (800,)
y_test: (200,)

Feature types:
Numerical: 4
Categorical: 5

Model-specific data:
CatBoost: (800, 9)
XGBoost: (800, 16)
LightGBM: (800, 16)
TabPFN: (800, 9)
